In [29]:
!pip3 install pandas transformers torch pillow image==1.5.33 requests==2.32.0 --break-system-packages

### Image Preparation

In [30]:
url_image_1 = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/5uo16pKhdB1f2Vz7H8Utkg/image-1.png'
url_image_2 = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/fsuegY1q_OxKIxNhf6zeYg/image-2.png'
url_image_3 = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/KCh_pM9BVWq_ZdzIBIA9Fw/image-3.png'
url_image_4 = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/VaaYLw52RaykwrE3jpFv7g/image-4.png'

image_urls = [url_image_1, url_image_2, url_image_3, url_image_4] 

In [31]:
import pandas as pd
from IPython.display import display, HTML

df = pd.DataFrame({
    "name": ["1", "2", "3", "4"],
    "image_url": image_urls
})

df["image"] = df["image_url"].apply(
    lambda u: f'<img src="{u}" width="350" style="border-radius:6px;" />'
)

display(HTML(df[["name", "image"]].to_html(escape=False, index=False)))


name,image
1,
2,
3,
4,


##### Encoding the Images

In [42]:
import base64
import requests

def encode_images_to_base64(image_urls):
    """
    Downloads and encodes a list of image URLs to base64 strings.

    Parameters:
    - image_urls (list): A list of image URLs.

    Returns:
    - list: A list of base64-encoded image strings.
    """
    encoded_images = []
    for url in image_urls:
        response = requests.get(url)
        if response.status_code == 200:
            encoded_image = base64.b64encode(response.content).decode("utf-8")
            encoded_images.append(encoded_image)
            print(type(encoded_image))
        else:
            print(f"Warning: Failed to fetch image from {url} (Status code: {response.status_code})")
            encoded_images.append(None)
    return encoded_images

In [43]:
encoded_images = encode_images_to_base64(image_urls)

<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>


### Define the Model

##### Understanding how VL (Visual Learning) Models Works

<img src="https://blog.roboflow.com/content/images/2025/02/vlm.png" width="700" />

For example, the Florence-2 model,

<img src="https://blog.roboflow.com/content/images/2025/02/florence.png" width="600" />

In [51]:
import base64
from io import BytesIO
from PIL import Image
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq

# Load once
model_id = "Qwen/Qwen3-VL-2B-Instruct"
device = "cpu"
dtype = torch.float32

model = AutoModelForVision2Seq.from_pretrained(
    model_id,
    torch_dtype=dtype,
    low_cpu_mem_usage=True
).to(device)

In [57]:
def generate_model_response(encoded_image, user_query, assistant_prompt="You are a helpful assistant. Answer in 1 or 2 sentences.", max_new_tokens=512):
    img_bytes = base64.b64decode(encoded_image.split(",")[-1])
    image = Image.open(BytesIO(img_bytes)).convert("RGB")

    messages = [
        {"role": "system", "content": [{"type": "text", "text": assistant_prompt}]},
        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_query}]},
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

    # keep only new tokens (assistant output), not prompt tokens
    new_tokens = generated_ids[:, inputs["input_ids"].shape[1]:]
    answer = processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()
    return answer


### Multimodal Inference Function

The function sends an image and a query to the AI model and retrieves a description or answer. It combines a text-based prompt and an image to guide the model in generating a concise response.

- **`encoded_image`** (`str`): A base64-encoded image string, which allows the model to process the image data.

- **`user_query`** (`str`): The user's question about the image, providing context for the model to interpret the image and answer appropriately.

- **`assistant_prompt`** (`str`): An optional text prompt to guide the model in responding in a specific way. By default, the prompt is set to: `"You are a helpful assistant. Answer the following user query in 1 or 2 sentences:"`.

In [58]:
user_query = "Describe the photo"

for i in range(len(encoded_images)):
    image = encoded_images[i]

    response = generate_model_response(image, user_query)

    # Print the response with a formatted description
    print(f"Description for image {i + 1}: {response}")

Description for image 1: This is a vibrant, wide-angle street view of a busy city street, likely in New York City, with tall buildings on both sides, many of which are modern skyscrapers. The street is filled with cars, pedestrians, and yellow taxis, with traffic lights and street signs visible. A prominent sign for "thenomoma.com" is on a building in the distance, and another sign for "The Vanguard" is on a building to the right. The scene is bright and sunny, with trees providing some greenery along the sidewalk.
Description for image 2: A woman in a yellow jacket is running on an asphalt road in front of a large white building labeled "BLOCK II" and "5-7".
Description for image 3: This is an aerial photograph of a rural farmstead, likely in the United States, that has been severely flooded. The scene shows a house and several silos surrounded by water, with a large, flooded field to the left. The farm is situated in a low-lying area, with the water level reaching the base of the bui

##### Object Detection

In [59]:
image = encoded_images[1]

user_query = "How many cars are in this image?"

print("User Query: ", user_query)
print("Model Response: ", generate_model_response(image, user_query))

User Query:  How many cars are in this image?
Model Response:  1


In [60]:
image = encoded_images[2]

user_query = "How severe is the damage in this image?"

print("User Query: ", user_query)
print("Model Response: ", generate_model_response(image, user_query))

User Query:  How severe is the damage in this image?
Model Response:  The damage is severe, with the entire area flooded and the buildings and farm equipment submerged.


In [61]:
image = encoded_images[3]

user_query = "How much sodium in mg is in this product?"

print("User Query: ", user_query)
print("Model Response: ", generate_model_response(image, user_query))

User Query:  How much sodium in mg is in this product?
Model Response:  640mg
